# Assignment 1: Bi Encoder retrieval vs Cross Encoder re ranking

Document: the Catan base game rulebook (`catan_rules.pdf`, 16 pages).
Pipeline 1 takes the top 3 chunks from a bi encoder search. Pipeline 2 takes the top 10 from the same search and re ranks them with a cross encoder.
Models: `all-MiniLM-L6-v2` (bi encoder), `ms-marco-MiniLM-L-6-v2` (cross encoder), `Qwen2.5-1.5B-Instruct` (LLM, runs locally on the cluster).

In [1]:
import time
import numpy as np
import pandas as pd
import pymupdf
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer

## 1. Load the PDF and split it into chunks of 120 words (same chunks for both pipelines)

In [2]:
words = []
for page in pymupdf.open("catan_rules.pdf"):
    for block in page.get_text("blocks"):
        words += block[4].split()
chunks = [" ".join(words[i:i + 120]) for i in range(0, len(words), 120)]
print(len(chunks), "chunks")

86 chunks


## 2. Models and vector store

The vector store is a matrix with one normalised embedding per chunk, so the inner product of a query vector with it is the cosine similarity.

In [3]:
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
llm_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name)

vectors = bi_encoder.encode(chunks, normalize_embeddings=True)

## 3. The two pipelines

In [4]:
def baseline_top10(query):
    scores = vectors @ bi_encoder.encode(query, normalize_embeddings=True)
    return list(np.argsort(-scores)[:10])

def rerank(query, candidates):
    scores = cross_encoder.predict([(query, chunks[i]) for i in candidates])
    return [candidates[i] for i in np.argsort(-scores)]

def answer(query, chunk_ids):
    context = "\n\n".join(chunks[i] for i in chunk_ids)
    messages = [{"role": "user", "content": f"Answer the question using only the rulebook excerpts below.\n\n{context}\n\nQuestion: {query}"}]
    ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    out = llm.generate(**ids, max_new_tokens=120, do_sample=False)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

## 4. Failure mode queries
Each query has a key phrase. The key chunk is the chunk that contains this phrase, so we can see at which rank each pipeline puts it.

In [5]:
queries = [
 [
  "Exception",
  "If two players tie for longest road, who holds the card?",
  "two or more players tie for the new longest road"
 ],
 [
  "Buried chunk",
  "When can victory point cards be shown?",
  "You only reveal victory point cards when the game is over"
 ],
 [
  "Keyword trap",
  "Who goes first in the game?",
  "the oldest player goes first"
 ],
 [
  "Negation",
  "Is it forbidden to reveal victory point cards early?",
  "You only reveal victory point cards when the game is over"
 ],
 [
  "Buried chunk",
  "Who can steal a card when the robber is moved?",
  "you also steal 1 resource card at random"
 ],
 [
  "Exception, both fail",
  "Can I play the development card I just bought?",
  "may not be a card you bought during the same turn"
 ],
 [
  "Control",
  "Do I lose cards when someone else rolls a seven and I only have six?",
  "has only 6 cards in his hand"
 ]
]

In [6]:
rows = []
for label, query, key in queries:
    key_chunk = next(i for i, c in enumerate(chunks) if key in c)

    t = time.time()
    top10 = baseline_top10(query)
    t_base = time.time() - t

    t = time.time()
    reranked = rerank(query, top10)
    t_rerank = time.time() - t

    rank1 = top10.index(key_chunk) + 1 if key_chunk in top10 else None
    rank2 = reranked.index(key_chunk) + 1 if key_chunk in reranked else None
    answer1 = answer(query, top10[:3])
    answer2 = answer(query, reranked[:3])
    rows.append(dict(label=label, query=query, key_chunk=key_chunk, rank_baseline=rank1, rank_rerank=rank2,
                     t_baseline=t_base, t_rerank=t_rerank, answer1=answer1, answer2=answer2))
    print(f"[{label}] {query}\n  key chunk {key_chunk}: baseline rank {rank1} -> reranked rank {rank2}")
    print("  Answer 1:", answer1.replace("\n", " "))
    print("  Answer 2:", answer2.replace("\n", " "))

[Exception] If two players tie for longest road, who holds the card?
  key chunk 39: baseline rank 5 -> reranked rank 1
  Answer 1: If two players tie for the longest road, neither player will hold the "Longest Road" card because each player would need to have built a road that was exactly as long as the other's longest road to be eligible for the card. The rules state that if there is a tie for the longest road, no one wins the card; instead, both players receive a 1 victory point each.
  Answer 2: If two players tie for the longest road, both players would hold the "Longest Road" card. According to the rulebook excerpt provided:  "If you no longer have the longest road, but two or more players tie for the new longest road, set the 'Longest Road' card aside."  This means that once there's a tie for the longest road, the cards remain with those players until someone breaks the tie.


[Buried chunk] When can victory point cards be shown?
  key chunk 28: baseline rank 9 -> reranked rank 2
  Answer 1: Victory point cards can be revealed when you or someone else wins the game, or if you have exactly 10 victory points at the end of your turn.
  Answer 2: Victory point cards can be revealed when either:  1. During your own turn, specifically when you reach exactly 10 victory points. 2. After revealing them yourself, if you're confident you have 10 or more victory points. 3. When another player wins the game, even if they don't reach 10 points yet.


[Keyword trap] Who goes first in the game?
  key chunk 9: baseline rank 5 -> reranked rank 1
  Answer 1: The highest-ranking player starts the game as the starting player and begins.
  Answer 2: The highest player goes first in the game.


[Negation] Is it forbidden to reveal victory point cards early?
  key chunk 28: baseline rank 4 -> reranked rank 3
  Answer 1: Yes, according to the rulebook excerpt provided, victory point cards must be kept hidden until the end of the game unless explicitly stated otherwise. The rule states "Keep victory point cards hidden until the end of the game." Therefore, revealing victory point cards before this time would be against the rules.
  Answer 2: Yes, according to the rulebook excerpt provided, victory point cards must be kept hidden until the end of the game unless revealed due to reaching exactly 10 victory points or being declared as the winner. Therefore, it is not allowed to reveal victory point cards early under normal circumstances.


[Buried chunk] Who can steal a card when the robber is moved?
  key chunk 52: baseline rank 4 -> reranked rank 2
  Answer 1: When the robber is moved, any player who has a settlement or city adjacent to the target terrain hex can steal a card from the player who was robbed. However, if the target hex is adjacent to two or more players' settlements or cities, the player chooses which one to steal from.
  Answer 2: When the robber is moved, a player who has a settlement or city adjacent to the target terrain hex receives a resource card stolen from them.


[Exception, both fail] Can I play the development card I just bought?
  key chunk 18: baseline rank None -> reranked rank None
  Answer 1: Yes, you can play the development card you just bought as long as it meets the criteria for playing a development card according to the rules provided. Specifically:  - You can play 1 development card per turn. - The card must be played after rolling the dice. - You cannot play a card that was bought during the same turn. - A victory point card Y that brings you to a certain number of points can also be used as a development card under specific conditions.  So, once you have met all these requirements, you can indeed play the card you purchased.
  Answer 2: Yes, you can play the development card you just bought as long as it meets the conditions for playing a development card according to the rules provided. Specifically:  - You can play the card at any time, including before rolling the dice. - However, you cannot play a card that was bought during

[Control] Do I lose cards when someone else rolls a seven and I only have six?
  key chunk 51: baseline rank 1 -> reranked rank 1
  Answer 1: Yes, according to the rulebook excerpt provided, if you roll a "7" and have fewer than 7 resource cards, you will need to discard half of your remaining cards. In this case, since you only have 6 cards, you would discard 3 cards (half of 6), returning the discarded cards back into the supply stack.
  Answer 2: Yes, according to the rulebook excerpt provided, if you roll a "7" and have fewer than 7 resource cards, you will need to discard half of those cards. In this case, since you only have 6 cards, you would discard 3 cards (half of 6 rounded down), returning the remaining 3 cards to the supply stack.


## 5. Rank shifts

In [7]:
df = pd.DataFrame(rows)
df[["label", "query", "key_chunk", "rank_baseline", "rank_rerank"]]

,label,query,key_chunk,rank_baseline,rank_rerank
0,Exception,"If two players tie for longest road, who holds...",39,5.0,1.0
1,Buried chunk,When can victory point cards be shown?,28,9.0,2.0
2,Keyword trap,Who goes first in the game?,9,5.0,1.0
3,Negation,Is it forbidden to reveal victory point cards ...,28,4.0,3.0
4,Buried chunk,Who can steal a card when the robber is moved?,52,4.0,2.0
5,"Exception, both fail",Can I play the development card I just bought?,18,NaN,NaN
6,Control,Do I lose cards when someone else rolls a seve...,51,1.0,1.0


## 6. Latency (seconds per query, retrieval only, LLM generation excluded)

In [8]:
latency = pd.DataFrame({
    "baseline (bi encoder search)": df.t_baseline,
    "re ranking pipeline (search + cross encoder on 10 pairs)": df.t_baseline + df.t_rerank,
})
latency.index = df.label
latency.loc["mean"] = latency.mean()
latency.round(4)

,baseline (bi encoder search),re ranking pipeline (search + cross encoder on 10 pairs)
label,,
Exception,0.0062,0.0708
Buried chunk,0.0106,0.0741
Keyword trap,0.0061,0.0598
Negation,0.0105,0.0763
Buried chunk,0.0064,0.0617
"Exception, both fail",0.0094,0.0750
Control,0.0077,0.0646
mean,0.0081,0.0689
